<a href="https://colab.research.google.com/github/lestojas/segmentation/blob/claude/lestojas-segmentation-direction-o9c5p8/colab/train_crack_direction_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Crack Detection, Segmentation & Direction Classification

This notebook trains and evaluates a two-model pipeline on the `lestojas/segmentation`
crack dataset, using **YOLO11** (Ultralytics' current generation — the most advanced
real-time detection/segmentation/classification architecture that can be trained
end-to-end in a single Colab session):

1. **YOLO11-seg** — instance segmentation model. Answers *"is there a crack, and what
   shape is it?"* (crack vs. no-crack detection + pixel-accurate mask segmentation).
2. **YOLO11-cls** — image classification model. Answers *"which direction does the
   crack run?"* (Horizontal / Vertical / Diagonal / Mixed), trained on the direction
   labels already computed in the repo (`*/_direction_labels.csv`, derived via PCA on
   the ground-truth segmentation polygons — see `DIRECTION_LABELS.md`).

A final evaluation section reports all three requested accuracies:
- **Crack vs. no-crack detection accuracy**
- **Crack shape segmentation accuracy** (mask IoU / mAP)
- **Crack direction classification accuracy**

Section 10 then packages the key results into **three research-paper-ready tables**
(Table 1: dataset composition, Table 2: descriptive performance metrics, Table 3: a
significance test of whether each task beats a naive baseline) — exported as CSV and
LaTeX and zipped for download.

> **⚠️ Dataset caveat (read before trusting the detection number):** every image in
> this dataset contains a crack — there are no genuine crack-free "negative" images.
> So the "crack vs. no-crack" metric below is really measuring **recall** (how often
> a crack that's there gets detected), not true accuracy against real negatives. If
> you want a rigorous no-crack accuracy figure, add a batch of crack-free background
> images (with empty annotations) to each split before training.

**Before running:** `Runtime → Change runtime type → GPU` (T4 or better).

## 0. Setup

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU detected — go to Runtime > Change runtime type > GPU before continuing.')

In [ ]:
!pip install -q ultralytics scikit-learn seaborn

In [ ]:
# --- Configuration ---------------------------------------------------------
REPO_URL = 'https://github.com/lestojas/segmentation.git'
BRANCH   = 'claude/lestojas-segmentation-direction-o9c5p8'  # switch to 'main' once this branch is merged

SEG_MODEL   = 'yolo11m-seg.pt'   # s/m/l/x — bigger = more accurate, slower. m is a good default for Colab.
CLS_MODEL   = 'yolo11m-cls.pt'
IMG_SIZE    = 640
SEG_EPOCHS  = 100
CLS_EPOCHS  = 50
BATCH       = -1   # -1 = Ultralytics auto-picks the largest batch that fits in GPU memory
CONF_THRES  = 0.25 # confidence threshold used for the 'is a crack detected in this image' metric

SAVE_TO_DRIVE = False  # set True to copy trained weights to your Google Drive at the end

In [ ]:
import os, shutil
DATA_DIR = '/content/segmentation'
if os.path.isdir(DATA_DIR):
    shutil.rmtree(DATA_DIR)
!git clone --branch {BRANCH} --single-branch {REPO_URL} {DATA_DIR}
os.chdir(DATA_DIR)
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
import json, csv, math, random, zipfile
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageDraw
from sklearn.metrics import confusion_matrix, classification_report
from scipy import stats
from IPython.display import display

from ultralytics import YOLO

SPLITS = ['train', 'valid', 'test']
random.seed(0)

## 1. Convert COCO segmentation annotations → YOLO-seg format

Ultralytics expects one `.txt` label file per image, one line per instance:
`class_id x1 y1 x2 y2 ... xn yn` with all coordinates normalized to `[0, 1]`. This
dataset has a single object class (`Cracks`), so every instance maps to class `0`.

In [ ]:
YOLO_SEG_DIR = Path('/content/yolo_seg_dataset')
if YOLO_SEG_DIR.exists():
    shutil.rmtree(YOLO_SEG_DIR)

def convert_split_to_yolo_seg(split):
    coco = json.load(open(f'{DATA_DIR}/{split}/_annotations.coco.json'))
    anns_by_image = defaultdict(list)
    for a in coco['annotations']:
        anns_by_image[a['image_id']].append(a)

    img_out = YOLO_SEG_DIR / split / 'images'
    lbl_out = YOLO_SEG_DIR / split / 'labels'
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for im in coco['images']:
        w, h = im['width'], im['height']
        src = Path(DATA_DIR) / split / im['file_name']
        dst = img_out / im['file_name']
        if not dst.exists():
            os.symlink(src, dst)

        lines = []
        for ann in anns_by_image.get(im['id'], []):
            for seg in ann.get('segmentation', []):
                if len(seg) < 6:
                    continue
                coords = []
                for i in range(0, len(seg), 2):
                    x = min(max(seg[i] / w, 0.0), 1.0)
                    y = min(max(seg[i + 1] / h, 0.0), 1.0)
                    coords.append(f'{x:.6f} {y:.6f}')
                lines.append('0 ' + ' '.join(coords))

        (lbl_out / (Path(im['file_name']).stem + '.txt')).write_text('\n'.join(lines))

    return len(coco['images']), sum(len(v) for v in anns_by_image.values())

for split in SPLITS:
    n_img, n_ann = convert_split_to_yolo_seg(split)
    print(f'{split}: {n_img} images, {n_ann} annotations converted')

In [ ]:
data_yaml = f'''
path: {YOLO_SEG_DIR}
train: train/images
val: valid/images
test: test/images
names:
  0: Cracks
'''
yaml_path = YOLO_SEG_DIR / 'data.yaml'
yaml_path.write_text(data_yaml)
print(data_yaml)

## 2. Build the direction-classification dataset

Ultralytics' classification trainer expects `train/<class_name>/*.jpg`,
`val/<class_name>/*.jpg` folders. We build that structure from the
`_direction_labels.csv` files already computed for this dataset (see
`DIRECTION_LABELS.md` for how those labels were derived).

In [ ]:
DIR_DATASET = Path('/content/direction_dataset')
if DIR_DATASET.exists():
    shutil.rmtree(DIR_DATASET)

SPLIT_TO_YOLO_CLS = {'train': 'train', 'valid': 'val', 'test': 'test'}
direction_rows = []

for split in SPLITS:
    with open(f'{DATA_DIR}/{split}/_direction_labels.csv') as f:
        rows = list(csv.DictReader(f))
    out_split = SPLIT_TO_YOLO_CLS[split]
    for r in rows:
        cls_dir = DIR_DATASET / out_split / r['direction']
        cls_dir.mkdir(parents=True, exist_ok=True)
        src = Path(DATA_DIR) / split / r['file_name']
        dst = cls_dir / r['file_name']
        if not dst.exists():
            os.symlink(src, dst)
        r['split'] = split
        direction_rows.append(r)

direction_df = pd.DataFrame(direction_rows)
direction_df['num_annotations'] = direction_df['num_annotations'].astype(int)
direction_df['angle_deg'] = pd.to_numeric(direction_df['angle_deg'], errors='coerce')
direction_df['elongation_ratio'] = pd.to_numeric(direction_df['elongation_ratio'], errors='coerce')
print(direction_df.groupby(['split', 'direction']).size().unstack(fill_value=0))

## 3. Train YOLO11-seg — crack detection + shape segmentation

In [ ]:
seg_model = YOLO(SEG_MODEL)
seg_train_results = seg_model.train(
    data=str(yaml_path),
    epochs=SEG_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=20,
    project='runs',
    name='crack_seg',
    seed=0,
)

## 4. Train YOLO11-cls — crack direction classification

In [ ]:
cls_model = YOLO(CLS_MODEL)
cls_train_results = cls_model.train(
    data=str(DIR_DATASET),
    epochs=CLS_EPOCHS,
    imgsz=224,
    batch=BATCH if BATCH != -1 else 64,
    patience=20,
    project='runs',
    name='crack_direction_cls',
    seed=0,
)

## 5. Evaluate — crack vs. no-crack detection accuracy

`seg_model.val()` gives standard object-detection metrics on the held-out **test**
split (box precision/recall/mAP). We also compute an image-level number: for each
test image, did the model fire at least one crack detection above `CONF_THRES`?
Since (per the caveat above) essentially every test image truly contains a crack,
this number is effectively the model's **crack recall / miss rate**, reported here
as "detection rate".

In [ ]:
best_seg = YOLO('runs/crack_seg/weights/best.pt')
seg_val = best_seg.val(data=str(yaml_path), split='test', imgsz=IMG_SIZE)

box_map50    = seg_val.box.map50
box_map50_95 = seg_val.box.map
box_precision = seg_val.box.mp
box_recall    = seg_val.box.mr
print(f'Box detection  — mAP50: {box_map50:.4f}  mAP50-95: {box_map50_95:.4f}  '
      f'Precision: {box_precision:.4f}  Recall: {box_recall:.4f}')

In [ ]:
# Image-level 'is there a detected crack in this image?' check
test_images_dir = YOLO_SEG_DIR / 'test' / 'images'
test_files = sorted(test_images_dir.glob('*.jpg'))

coco_test = json.load(open(f'{DATA_DIR}/test/_annotations.coco.json'))
gt_has_crack = defaultdict(bool)
for a in coco_test['annotations']:
    gt_has_crack[a['image_id']] = True
filename_to_gt = {im['file_name']: gt_has_crack.get(im['id'], False) for im in coco_test['images']}

y_true, y_pred = [], []
preds = best_seg.predict(source=[str(p) for p in test_files], conf=CONF_THRES, imgsz=IMG_SIZE, verbose=False)
for p in preds:
    fname = Path(p.path).name
    y_true.append(filename_to_gt.get(fname, False))
    y_pred.append(len(p.boxes) > 0)

y_true = np.array(y_true); y_pred = np.array(y_pred)
detection_accuracy = (y_true == y_pred).mean()
n_pos = y_true.sum(); n_neg = (~y_true).sum()
print(f'Test images: {len(y_true)}  (crack-labeled: {n_pos}, crack-free: {n_neg})')
print(f'Image-level crack detection accuracy: {detection_accuracy:.4f}')
if n_pos > 0:
    print(f'  Crack recall (of {n_pos} true-crack images, detected): {(y_pred[y_true]).mean():.4f}')
if n_neg > 0:
    print(f'  True-negative rate (of {n_neg} crack-free images, correctly said none): {(~y_pred[~y_true]).mean():.4f}')
else:
    print('  (No crack-free test images exist in this dataset — true-negative rate is undefined; see caveat in Section 0.)')

## 6. Evaluate — crack shape segmentation accuracy

Mask mAP/precision/recall come straight out of the same `val()` call (segmentation
metrics are computed alongside detection metrics for a `-seg` model). We also
compute the mean IoU between each predicted mask and its best-matching ground-truth
mask, which is a more intuitive "how good is the crack shape" number.

In [ ]:
seg_map50    = seg_val.seg.map50
seg_map50_95 = seg_val.seg.map
seg_precision = seg_val.seg.mp
seg_recall    = seg_val.seg.mr
print(f'Mask segmentation — mAP50: {seg_map50:.4f}  mAP50-95: {seg_map50_95:.4f}  '
      f'Precision: {seg_precision:.4f}  Recall: {seg_recall:.4f}')

In [ ]:
def polygon_mask(segmentation, w, h):
    mask = Image.new('L', (w, h), 0)
    draw = ImageDraw.Draw(mask)
    for seg in segmentation:
        pts = list(zip(seg[0::2], seg[1::2]))
        if len(pts) >= 3:
            draw.polygon(pts, fill=1)
    return np.array(mask, dtype=bool)

def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return inter / union if union > 0 else 0.0

id_by_name = {im['file_name']: im for im in coco_test['images']}
anns_by_image_test = defaultdict(list)
for a in coco_test['annotations']:
    anns_by_image_test[a['image_id']].append(a)

ious = []
pred_masks_iter = best_seg.predict(source=[str(p) for p in test_files], conf=CONF_THRES, imgsz=IMG_SIZE, verbose=False)
for p in pred_masks_iter:
    fname = Path(p.path).name
    im_meta = id_by_name.get(fname)
    if im_meta is None:
        continue
    w, h = im_meta['width'], im_meta['height']
    gt_masks = [polygon_mask(a['segmentation'], w, h) for a in anns_by_image_test.get(im_meta['id'], [])]
    if not gt_masks:
        continue
    if p.masks is None:
        ious.extend([0.0] * len(gt_masks))  # missed every ground-truth crack in this image
        continue
    # p.masks.xy gives polygon vertices already in original-image pixel coordinates,
    # which avoids any ambiguity about the resolution of p.masks.data.
    pred_masks = [polygon_mask([poly.reshape(-1).tolist()], w, h) for poly in p.masks.xy]
    used = set()
    for gt in gt_masks:
        best_iou, best_j = 0.0, None
        for j, pm in enumerate(pred_masks):
            if j in used:
                continue
            iou = mask_iou(gt, pm)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_j is not None:
            used.add(best_j)
        ious.append(best_iou)

ious = np.array(ious)
print(f'Matched {len(ious)} ground-truth crack instances across {len(test_files)} test images')
print(f'Mean mask IoU: {ious.mean():.4f}   Median: {np.median(ious):.4f}')
print(f'Instances with IoU > 0.5 (good shape match): {(ious > 0.5).mean():.4f}')

plt.figure(figsize=(6, 4))
plt.hist(ious, bins=20, edgecolor='black')
plt.xlabel('Mask IoU (predicted vs. ground truth)'); plt.ylabel('Number of crack instances')
plt.title('Segmentation shape accuracy — IoU distribution'); plt.tight_layout(); plt.show()

## 7. Evaluate — crack direction classification accuracy

In [ ]:
best_cls = YOLO('runs/crack_direction_cls/weights/best.pt')
cls_val = best_cls.val(data=str(DIR_DATASET), split='test')
print(f'Top-1 accuracy: {cls_val.top1:.4f}   Top-5 accuracy: {cls_val.top5:.4f}')

In [ ]:
class_names = best_cls.names  # {index: class_name}
name_to_idx = {v: k for k, v in class_names.items()}

test_dir_dataset = DIR_DATASET / 'test'
y_true_dir, y_pred_dir, files = [], [], []
for cls_name in sorted(os.listdir(test_dir_dataset)):
    for f in (test_dir_dataset / cls_name).glob('*.jpg'):
        files.append(f)
        y_true_dir.append(cls_name)

preds = best_cls.predict(source=[str(f) for f in files], imgsz=224, verbose=False)
for p in preds:
    y_pred_dir.append(class_names[int(p.probs.top1)])

print(classification_report(y_true_dir, y_pred_dir, digits=4))

labels_order = sorted(set(y_true_dir) | set(y_pred_dir))
cm = confusion_matrix(y_true_dir, y_pred_dir, labels=labels_order)
plt.figure(figsize=(5.5, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels_order, yticklabels=labels_order)
plt.xlabel('Predicted direction'); plt.ylabel('True direction')
plt.title('Direction classification — confusion matrix'); plt.tight_layout(); plt.show()

## 8. Combined summary

In [ ]:
direction_accuracy = (np.array(y_true_dir) == np.array(y_pred_dir)).mean()

summary = pd.DataFrame([
    {'Task': 'Crack vs. no-crack detection', 'Metric': 'Image-level accuracy', 'Value': detection_accuracy},
    {'Task': 'Crack vs. no-crack detection', 'Metric': 'Box mAP50', 'Value': box_map50},
    {'Task': 'Crack vs. no-crack detection', 'Metric': 'Box mAP50-95', 'Value': box_map50_95},
    {'Task': 'Crack shape segmentation', 'Metric': 'Mean mask IoU', 'Value': ious.mean()},
    {'Task': 'Crack shape segmentation', 'Metric': 'Mask mAP50', 'Value': seg_map50},
    {'Task': 'Crack shape segmentation', 'Metric': 'Mask mAP50-95', 'Value': seg_map50_95},
    {'Task': 'Crack direction classification', 'Metric': 'Top-1 accuracy', 'Value': direction_accuracy},
])
summary['Value'] = summary['Value'].map(lambda v: f'{v:.4f}')
summary

## 9. Visualize sample predictions

A handful of test images with the predicted crack mask overlaid and the predicted
direction label as the title, for a quick sanity check.

In [ ]:
sample_files = random.sample(test_files, min(6, len(test_files)))
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, fpath in zip(axes.ravel(), sample_files):
    seg_pred = best_seg.predict(source=str(fpath), conf=CONF_THRES, imgsz=IMG_SIZE, verbose=False)[0]
    dir_pred = best_cls.predict(source=str(fpath), imgsz=224, verbose=False)[0]
    annotated = seg_pred.plot()[:, :, ::-1]  # BGR -> RGB
    ax.imshow(annotated)
    ax.set_title(f'{fpath.name}\npredicted direction: {class_names[int(dir_pred.probs.top1)]} '
                 f'({float(dir_pred.probs.top1conf):.2f})', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 10. Export research-paper-ready tables (descriptive & inferential statistics)

Three tables, covering the essentials only. Each is built as a `pandas` DataFrame,
displayed inline, and then written to `/content/paper_tables/` as both `.csv`
(spreadsheet-ready) and `.tex` (a LaTeX `tabular` you can `\input{}` straight into a
paper), bundled into `paper_tables.zip` for download.

- **Table 1 (descriptive)** — dataset composition: image/annotation counts and crack
  size by split and direction.
- **Table 2 (descriptive)** — one consolidated model performance table: accuracy,
  precision, recall, F1/mAP for all three tasks (detection, segmentation, direction).
- **Table 3 (inferential)** — for each task, a single significance test asking the
  most intuitive version of "is this real?": *is the model's accuracy significantly
  better than the naive/trivial baseline anyone could get without training anything?*

In [ ]:
PAPER_TABLES_DIR = Path('/content/paper_tables')
PAPER_TABLES_DIR.mkdir(exist_ok=True)
exported_tables = {}  # name -> DataFrame; exported to CSV/LaTeX in the final cell of this section

def register_table(name, df):
    exported_tables[name] = df
    return df

### Table 1 — Dataset descriptive statistics

Image and crack-instance counts, and crack size (as % of image area) by split and
direction — the standard "dataset characteristics" table for a methods section.

In [ ]:
all_coco = {split: json.load(open(f'{DATA_DIR}/{split}/_annotations.coco.json')) for split in SPLITS}

area_records = []
for split, coco in all_coco.items():
    imgs_by_id = {im['id']: im for im in coco['images']}
    for a in coco['annotations']:
        im = imgs_by_id[a['image_id']]
        area_records.append({
            'split': split,
            'direction': im.get('direction', 'Unknown'),
            'area_px': a['area'],
            'area_pct': 100 * a['area'] / (im['width'] * im['height']),
        })
area_df = pd.DataFrame(area_records)

image_counts = direction_df.groupby(['split', 'direction']).size().rename('n_images').reset_index()
ann_counts = (direction_df.groupby(['split', 'direction'])['num_annotations'].sum()
              .rename('n_annotations').reset_index())
area_stats = (area_df.groupby(['split', 'direction'])['area_pct']
              .agg(['count', 'mean', 'std', 'median', 'min', 'max']).reset_index())
area_stats.columns = ['split', 'direction', 'n_instances', 'mean_area_pct', 'sd_area_pct',
                       'median_area_pct', 'min_area_pct', 'max_area_pct']

table1 = (image_counts.merge(ann_counts, on=['split', 'direction'])
          .merge(area_stats, on=['split', 'direction']).round(3))
register_table('Table1_dataset_descriptives', table1)
display(table1)

### Table 2 — Model performance summary (descriptive)

One row per metric, grouped by task, so the whole results section is a single table:
accuracy/recall and box mAP for detection, mean IoU and mask mAP for segmentation,
and top-1 accuracy plus macro-averaged precision/recall/F1 for direction
classification.

In [ ]:
report_dict = classification_report(y_true_dir, y_pred_dir, output_dict=True, digits=4)
macro = report_dict['macro avg']
y_true_dir_arr, y_pred_dir_arr = np.array(y_true_dir), np.array(y_pred_dir)
dir_acc = (y_true_dir_arr == y_pred_dir_arr).mean()

table2 = pd.DataFrame([
    {'Task': 'Crack detection', 'Metric': 'Image-level accuracy', 'Value': detection_accuracy},
    {'Task': 'Crack detection', 'Metric': 'Recall (of true-crack images)',
     'Value': float(y_pred[y_true].mean()) if n_pos > 0 else np.nan},
    {'Task': 'Crack detection', 'Metric': 'Box precision', 'Value': box_precision},
    {'Task': 'Crack detection', 'Metric': 'Box recall', 'Value': box_recall},
    {'Task': 'Crack detection', 'Metric': 'Box mAP50', 'Value': box_map50},
    {'Task': 'Crack detection', 'Metric': 'Box mAP50-95', 'Value': box_map50_95},
    {'Task': 'Crack segmentation', 'Metric': 'Mean mask IoU', 'Value': ious.mean()},
    {'Task': 'Crack segmentation', 'Metric': 'Mask precision', 'Value': seg_precision},
    {'Task': 'Crack segmentation', 'Metric': 'Mask recall', 'Value': seg_recall},
    {'Task': 'Crack segmentation', 'Metric': 'Mask mAP50', 'Value': seg_map50},
    {'Task': 'Crack segmentation', 'Metric': 'Mask mAP50-95', 'Value': seg_map50_95},
    {'Task': 'Direction classification', 'Metric': 'Top-1 accuracy', 'Value': dir_acc},
    {'Task': 'Direction classification', 'Metric': 'Macro precision', 'Value': macro['precision']},
    {'Task': 'Direction classification', 'Metric': 'Macro recall', 'Value': macro['recall']},
    {'Task': 'Direction classification', 'Metric': 'Macro F1', 'Value': macro['f1-score']},
]).round(4)
register_table('Table2_performance_summary', table2)
display(table2)

### Table 3 — Is the model actually better than a naive guess? (inferential)

For each task, this compares the model's accuracy against the accuracy a **naive
baseline** would get with no training at all, using a one-sided significance test:

| Task | Naive baseline | Test |
|---|---|---|
| Detection | Always predict "crack present" | one-sided binomial test |
| Segmentation | Fixed cutoff: IoU ≥ 0.5 counts as an acceptable shape match | one-sided one-sample t-test |
| Direction | Always predict the most common direction class | one-sided binomial test |

**How to read the result:** `p < 0.05` means the model's accuracy is very unlikely to
be explained by the naive baseline alone — it is doing real, statistically meaningful
work. `p >= 0.05` means we can't rule out that the naive baseline explains the result
just as well — the model's edge over guessing isn't statistically established (often a
sign the test set is too small, or the naive baseline was already very strong).

In [ ]:
n_test_img = len(y_true)

# Detection: compare against always predicting 'crack present' (the caveat in Section 0
# is exactly why this is the fair baseline here, not a 50/50 coin flip)
baseline_det_p = n_pos / n_test_img
det_test = stats.binomtest(int((y_true == y_pred).sum()), n_test_img, baseline_det_p, alternative='greater')

# Segmentation: compare mean IoU against a fixed 'acceptable overlap' cutoff
iou_test = stats.ttest_1samp(ious, 0.5, alternative='greater')

# Direction: compare against always predicting the most common direction class
majority_class, majority_count = Counter(y_true_dir).most_common(1)[0]
baseline_dir_p = majority_count / len(y_true_dir)
dir_test = stats.binomtest(int((y_true_dir_arr == y_pred_dir_arr).sum()), len(y_true_dir_arr),
                            baseline_dir_p, alternative='greater')

table3 = pd.DataFrame([
    {'Task': 'Crack detection', 'Model accuracy': detection_accuracy,
     'Naive baseline': round(baseline_det_p, 4), 'Baseline meaning': "always predict 'crack present'",
     'p_value': det_test.pvalue, 'Significantly better?': det_test.pvalue < 0.05},
    {'Task': 'Crack segmentation', 'Model accuracy': round(ious.mean(), 4),
     'Naive baseline': 0.5, 'Baseline meaning': 'acceptable shape overlap cutoff (IoU=0.5)',
     'p_value': iou_test.pvalue, 'Significantly better?': iou_test.pvalue < 0.05},
    {'Task': 'Direction classification', 'Model accuracy': dir_acc,
     'Naive baseline': round(baseline_dir_p, 4),
     'Baseline meaning': f"always predict most common direction ('{majority_class}')",
     'p_value': dir_test.pvalue, 'Significantly better?': dir_test.pvalue < 0.05},
]).round(4)
register_table('Table3_significance_vs_naive_baseline', table3)
display(table3)

### Export all three tables to CSV + LaTeX and download

In [ ]:
for name, df in exported_tables.items():
    df.to_csv(PAPER_TABLES_DIR / f'{name}.csv', index=False)
    try:
        (PAPER_TABLES_DIR / f'{name}.tex').write_text(
            df.to_latex(index=False, float_format='%.4f', na_rep='--'))
    except Exception as e:
        print(f'Could not export {name} to LaTeX ({e}); CSV was still written.')

print(f'Exported {len(exported_tables)} tables to {PAPER_TABLES_DIR}:')
for f in sorted(PAPER_TABLES_DIR.iterdir()):
    print(' ', f.name)

zip_path = '/content/paper_tables.zip'
with zipfile.ZipFile(zip_path, 'w') as zf:
    for f in PAPER_TABLES_DIR.iterdir():
        zf.write(f, arcname=f.name)

try:
    from google.colab import files as colab_files
    colab_files.download(zip_path)
except Exception as e:
    print(f'Automatic browser download not available in this environment ({e}). '
          f'The files are still saved at {PAPER_TABLES_DIR} and {zip_path}.')

## 11. (Optional) Save trained weights & tables to Google Drive

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    out_dir = '/content/drive/MyDrive/crack_models'
    os.makedirs(out_dir, exist_ok=True)
    shutil.copy('runs/crack_seg/weights/best.pt', f'{out_dir}/crack_seg_best.pt')
    shutil.copy('runs/crack_direction_cls/weights/best.pt', f'{out_dir}/crack_direction_cls_best.pt')
    tables_out_dir = f'{out_dir}/paper_tables'
    if os.path.isdir(tables_out_dir):
        shutil.rmtree(tables_out_dir)
    shutil.copytree(PAPER_TABLES_DIR, tables_out_dir)
    print(f'Saved weights and paper tables to {out_dir}')
else:
    print('SAVE_TO_DRIVE is False — skipping. Weights remain at runs/crack_seg/weights/best.pt '
          'and runs/crack_direction_cls/weights/best.pt, and exported tables remain at '
          f'{PAPER_TABLES_DIR} / /content/paper_tables.zip for this session.')